# 🤖 Stock Market Social Bot - Testing Notebook

This notebook provides comprehensive testing and demonstration of the Automated AI Stock Bot project.

## 📋 Project Overview

The Stock Market Social Bot is a fully automated system that:
- ✅ Fetches real-time stock data from multiple sources (Alpha Vantage, Finnhub, etc.)
- ✅ Generates engaging social media posts using AI (Google Gemini)
- ✅ Creates professional thumbnail images
- ✅ Posts automatically to Facebook every 6 hours
- ✅ Includes smart filtering and rate limiting

## 🎯 Testing Objectives

1. **Component Testing**: Verify each module works independently
2. **Integration Testing**: Test the complete workflow
3. **Data Validation**: Ensure data accuracy and reliability
4. **Performance Testing**: Monitor response times and efficiency
5. **Error Handling**: Test fallback mechanisms

---


In [2]:
# Import required libraries
import os
import sys
import json
from datetime import datetime, timezone
import pandas as pd
import matplotlib.pyplot as plt

# Add src directory to path
sys.path.append('src')

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print("Libraries imported successfully!")


Libraries imported successfully!


## 1. Test Stock Data Fetcher


In [8]:
"""
Stock data fetcher module using yfinance
Fetches real-time stock prices and calculates percentage changes
"""

import yfinance as yf
import pandas as pd
from datetime import datetime, timezone
import logging
from typing import Dict, Optional
try:
    from .mock_data import generate_mock_stock_data
except ImportError:
    from mock_data import generate_mock_stock_data

logger = logging.getLogger(__name__)

class StockFetcher:
    """Handles fetching stock data from Yahoo Finance"""
    
    def __init__(self):
        self.logger = logging.getLogger(__name__)
    
    def fetch_stock_data(self, ticker: str) -> Optional[Dict]:
        """
        Fetch latest stock data for a given ticker
        
        Args:
            ticker (str): Stock ticker symbol (e.g., 'AAPL')
            
        Returns:
            Dict: Stock data including price, change percentage, and timestamp
        """
        try:
            self.logger.info(f"\nFetching data for ticker: {ticker}\n")
            
            # Create yfinance ticker object with session
            import requests
            session = requests.Session()
            session.headers.update({
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            })
            
            stock = yf.Ticker(ticker, session=session)
            
            # Try different periods and intervals
            hist = None
            for period, interval in [("5d", "1d"), ("1d", "1h"), ("1d", "5m")]:
                try:
                    self.logger.info(f"Trying period={period}, interval={interval}")
                    hist = stock.history(period=period, interval=interval)
                    if not hist.empty:
                        break
                except Exception as e:
                    self.logger.warning(f"Failed with period={period}, interval={interval}: {e}")
                    continue
            
            if hist is None or hist.empty:
                self.logger.warning(f"No real data available for ticker: {ticker}, using mock data for testing")
                return generate_mock_stock_data(ticker)
            
            # Get latest and previous close prices
            latest_close = float(hist['Close'].iloc[-1])
            prev_close = float(hist['Close'].iloc[-2]) if len(hist) > 1 else latest_close
            
            # Calculate percentage change
            change_pct = ((latest_close - prev_close) / prev_close * 100) if prev_close != 0 else 0.0
            
            # Get additional info with error handling
            try:
                info = stock.info
                company_name = info.get('longName', ticker)
                market_cap = info.get('marketCap', 0)
                currency = info.get('currency', 'USD')
            except Exception as e:
                self.logger.warning(f"Could not fetch additional info for {ticker}: {e}")
                company_name = ticker
                market_cap = 0
                currency = 'USD'
            
            # Create result dictionary
            result = {
                'ticker': ticker,
                'company_name': company_name,
                'price': latest_close,
                'change_pct': change_pct,
                'change_amount': latest_close - prev_close,
                'previous_close': prev_close,
                'timestamp': datetime.now(timezone.utc),
                'volume': int(hist['Volume'].iloc[-1]) if 'Volume' in hist.columns else 0,
                'market_cap': market_cap,
                'currency': currency
            }
            
            self.logger.info(f"Successfully fetched data for {ticker}: ${latest_close:.2f} ({change_pct:+.2f}%)")
            return result
            
        except Exception as e:
            self.logger.warning(f"Error fetching real data for {ticker}: {str(e)}, using mock data for testing")
            return generate_mock_stock_data(ticker)
    
    def fetch_multiple_stocks(self, tickers: list) -> Dict[str, Dict]:
        """
        Fetch data for multiple tickers
        
        Args:
            tickers (list): List of ticker symbols
            
        Returns:
            Dict: Dictionary with ticker as key and stock data as value
        """
        results = {}
        for ticker in tickers:
            data = self.fetch_stock_data(ticker)
            if data:
                results[ticker] = data
        return results
    
    def is_significant_change(self, change_pct: float, threshold: float = 0.5) -> bool:
        """
        Check if the price change is significant enough to post
        
        Args:
            change_pct (float): Percentage change
            threshold (float): Minimum change threshold
            
        Returns:
            bool: True if change is significant
        """
        return abs(change_pct) >= threshold

# Example usage and testing
if __name__ == "__main__":
    # Set up logging
    logging.basicConfig(level=logging.INFO)
    
    # Test the fetcher
    fetcher = StockFetcher()
    
    # Test single ticker
    data = fetcher.fetch_stock_data("AAPL")
    if data:
        print(f"Stock Data: {data}")
        print(f"Significant change: {fetcher.is_significant_change(data['change_pct'])}")
    
    # Test multiple tickers
    tickers = ["AAPL", "MSFT", "GOOGL"]
    results = fetcher.fetch_multiple_stocks(tickers)
    print(f"\nMultiple stocks: {len(results)} fetched successfully")


INFO:__main__:
Fetching data for ticker: AAPL

INFO:__main__:Trying period=5d, interval=1d


ERROR:yfinance:Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:AAPL: No price data found, symbol may be delisted (period=5d)
INFO:__main__:Trying period=1d, interval=1h
ERROR:yfinance:Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:AAPL: No price data found, symbol may be delisted (period=1d)
INFO:__main__:Trying period=1d, interval=5m
ERROR:yfinance:Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:AAPL: No price data found, symbol may be delisted (period=1d)
INFO:__main__:
Fetching data for ticker: AAPL

INFO:__main__:Trying period=5d, interval=1d


Stock Data: {'ticker': 'AAPL', 'company_name': 'Apple Inc.', 'price': 201.33, 'change_pct': -3.24, 'change_amount': -6.75, 'previous_close': 208.08, 'timestamp': datetime.datetime(2025, 9, 26, 13, 40, 44, 484188, tzinfo=datetime.timezone.utc), 'volume': 1685495, 'market_cap': 121367591870, 'currency': 'USD'}
Significant change: True


ERROR:yfinance:Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:AAPL: No price data found, symbol may be delisted (period=5d)
INFO:__main__:Trying period=1d, interval=1h
ERROR:yfinance:Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:AAPL: No price data found, symbol may be delisted (period=1d)
INFO:__main__:Trying period=1d, interval=5m
ERROR:yfinance:Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:AAPL: No price data found, symbol may be delisted (period=1d)
INFO:__main__:
Fetching data for ticker: MSFT

INFO:__main__:Trying period=5d, interval=1d
ERROR:yfinance:Failed to get ticker 'MSFT' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:MSFT: No price data found, symbol may be delisted (period=5d)
INFO:__main__:Trying period=1d, interval=1h
ERROR:yfinance:Failed to get ticker 'MSFT' reason: Expecting value: line 1 column 1 (char 0)
ERRO


Multiple stocks: 3 fetched successfully


In [3]:
# Test the stock fetcher
from fetcher import StockFetcher

fetcher = StockFetcher()

# Test single ticker
ticker = "AAPL"
stock_data = fetcher.fetch_stock_data(ticker)

if stock_data:
    print(f"✅ Successfully fetched data for {ticker}")
    print(f"Company: {stock_data['company_name']}")
    print(f"Price: ${stock_data['price']:.2f}")
    print(f"Change: {stock_data['change_pct']:+.2f}%")
    print(f"Volume: {stock_data['volume']:,}")
    print(f"Timestamp: {stock_data['timestamp']}")
else:
    print(f"❌ Failed to fetch data for {ticker}")


Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
AAPL: No price data found, symbol may be delisted (period=5d)
Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
AAPL: No price data found, symbol may be delisted (period=1d)
Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
AAPL: No price data found, symbol may be delisted (period=1d)
No real data available for ticker: AAPL, using mock data for testing


✅ Successfully fetched data for AAPL
Company: Apple Inc.
Price: $435.16
Change: +3.00%
Volume: 4,171,718
Timestamp: 2025-09-26 12:34:52.561139+00:00


In [6]:
import os
import sys
import json
import time
import logging
import requests
import yfinance as yf
import pandas as pd
from datetime import datetime, timezone, timedelta
from typing import Dict, Optional, List, Tuple
from dataclasses import dataclass, asdict
from pathlib import Path
import sqlite3
from contextlib import contextmanager
import hashlib

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('stock_bot.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

@dataclass
class StockData:
    """Data structure for stock information"""
    ticker: str
    company_name: str
    price: float
    change_pct: float
    change_amount: float
    previous_close: float
    timestamp: datetime
    volume: int
    market_cap: int
    currency: str
    source: str
    
    def to_dict(self) -> Dict:
        data = asdict(self)
        data['timestamp'] = self.timestamp.isoformat()
        return data
    
    @classmethod
    def from_dict(cls, data: Dict) -> 'StockData':
        data['timestamp'] = datetime.fromisoformat(data['timestamp'])
        return cls(**data)

class DatabaseManager:
    """Handles SQLite database operations for storing stock data and bot state"""
    
    def __init__(self, db_path: str = "stock_bot.db"):
        self.db_path = db_path
        self.init_database()
    
    def init_database(self):
        """Initialize database tables"""
        with self.get_connection() as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS stock_data (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    ticker TEXT,
                    company_name TEXT,
                    price REAL,
                    change_pct REAL,
                    change_amount REAL,
                    previous_close REAL,
                    timestamp TEXT,
                    volume INTEGER,
                    market_cap INTEGER,
                    currency TEXT,
                    source TEXT,
                    created_at TEXT DEFAULT CURRENT_TIMESTAMP
                )
            """)
            
            conn.execute("""
                CREATE TABLE IF NOT EXISTS bot_posts (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    ticker TEXT,
                    content_hash TEXT,
                    content TEXT,
                    image_path TEXT,
                    platform TEXT,
                    posted_at TEXT,
                    success BOOLEAN,
                    error_message TEXT
                )
            """)
            
            conn.execute("""
                CREATE TABLE IF NOT EXISTS bot_state (
                    key TEXT PRIMARY KEY,
                    value TEXT,
                    updated_at TEXT DEFAULT CURRENT_TIMESTAMP
                )
            """)
    
    @contextmanager
    def get_connection(self):
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        try:
            yield conn
            conn.commit()
        except Exception as e:
            conn.rollback()
            raise e
        finally:
            conn.close()
    
    def save_stock_data(self, stock_data: StockData):
        """Save stock data to database"""
        with self.get_connection() as conn:
            conn.execute("""
                INSERT INTO stock_data 
                (ticker, company_name, price, change_pct, change_amount, previous_close, 
                 timestamp, volume, market_cap, currency, source)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                stock_data.ticker, stock_data.company_name, stock_data.price,
                stock_data.change_pct, stock_data.change_amount, stock_data.previous_close,
                stock_data.timestamp.isoformat(), stock_data.volume, stock_data.market_cap,
                stock_data.currency, stock_data.source
            ))
    
    def get_last_post_time(self, ticker: str) -> Optional[datetime]:
        """Get the last successful post time for a ticker"""
        with self.get_connection() as conn:
            result = conn.execute("""
                SELECT posted_at FROM bot_posts 
                WHERE ticker = ? AND success = 1 
                ORDER BY posted_at DESC LIMIT 1
            """, (ticker,)).fetchone()
            
            if result:
                return datetime.fromisoformat(result['posted_at'])
        return None
    
    def save_post_record(self, ticker: str, content: str, image_path: str, 
                        platform: str, success: bool, error_message: str = None):
        """Save post record to database"""
        content_hash = hashlib.md5(content.encode()).hexdigest()
        with self.get_connection() as conn:
            conn.execute("""
                INSERT INTO bot_posts 
                (ticker, content_hash, content, image_path, platform, posted_at, success, error_message)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                ticker, content_hash, content, image_path, platform,
                datetime.now(timezone.utc).isoformat(), success, error_message
            ))

class MultiSourceStockFetcher:
    """Robust stock data fetcher with multiple data sources and fallbacks"""
    
    def __init__(self):
        self.logger = logging.getLogger(self.__class__.__name__)
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
        
        # API keys from environment variables
        self.alpha_vantage_key = os.getenv('ALPHA_VANTAGE_API_KEY')
        self.finnhub_key = os.getenv('FINNHUB_API_KEY')
    
    def fetch_from_yfinance(self, ticker: str) -> Optional[StockData]:
        """Fetch data using yfinance (Yahoo Finance)"""
        try:
            self.logger.info(f"Fetching {ticker} from yfinance...")
            
            stock = yf.Ticker(ticker, session=self.session)
            
            # Try different periods
            hist = None
            for period, interval in [("5d", "1d"), ("2d", "1d"), ("1d", "1h")]:
                try:
                    hist = stock.history(period=period, interval=interval, timeout=10)
                    if not hist.empty:
                        break
                except Exception as e:
                    self.logger.warning(f"yfinance failed with {period}/{interval}: {e}")
                    continue
            
            if hist is None or hist.empty:
                return None
            
            # Get latest data
            latest_close = float(hist['Close'].iloc[-1])
            prev_close = float(hist['Close'].iloc[-2]) if len(hist) > 1 else latest_close
            change_pct = ((latest_close - prev_close) / prev_close * 100) if prev_close != 0 else 0.0
            
            # Get additional info with timeout
            try:
                info = stock.info
                company_name = info.get('longName', ticker)
                market_cap = info.get('marketCap', 0)
                currency = info.get('currency', 'USD')
            except:
                company_name = ticker
                market_cap = 0
                currency = 'USD'
            
            return StockData(
                ticker=ticker,
                company_name=company_name,
                price=latest_close,
                change_pct=change_pct,
                change_amount=latest_close - prev_close,
                previous_close=prev_close,
                timestamp=datetime.now(timezone.utc),
                volume=int(hist['Volume'].iloc[-1]) if 'Volume' in hist.columns else 0,
                market_cap=market_cap or 0,
                currency=currency,
                source='yfinance'
            )
            
        except Exception as e:
            self.logger.error(f"yfinance error for {ticker}: {e}")
            return None
    
    def fetch_from_alpha_vantage(self, ticker: str) -> Optional[StockData]:
        """Fetch data from Alpha Vantage API"""
        if not self.alpha_vantage_key:
            return None
            
        try:
            self.logger.info(f"Fetching {ticker} from Alpha Vantage...")
            
            url = f"https://www.alphavantage.co/query"
            params = {
                'function': 'GLOBAL_QUOTE',
                'symbol': ticker,
                'apikey': self.alpha_vantage_key
            }
            
            response = self.session.get(url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if 'Global Quote' not in data:
                return None
            
            quote = data['Global Quote']
            if not quote or '05. price' not in quote:
                return None
            
            price = float(quote['05. price'])
            change_pct = float(quote['10. change percent'].replace('%', ''))
            prev_close = float(quote['08. previous close'])
            
            return StockData(
                ticker=ticker,
                company_name=quote.get('01. symbol', ticker),
                price=price,
                change_pct=change_pct,
                change_amount=price - prev_close,
                previous_close=prev_close,
                timestamp=datetime.now(timezone.utc),
                volume=int(float(quote.get('06. volume', 0))),
                market_cap=0,  # Not provided by Alpha Vantage
                currency='USD',
                source='alpha_vantage'
            )
            
        except Exception as e:
            self.logger.error(f"Alpha Vantage error for {ticker}: {e}")
            return None
    
    def fetch_from_finnhub(self, ticker: str) -> Optional[StockData]:
        """Fetch data from Finnhub API"""
        if not self.finnhub_key:
            return None
            
        try:
            self.logger.info(f"Fetching {ticker} from Finnhub...")
            
            # Get current price
            url = f"https://finnhub.io/api/v1/quote"
            params = {'symbol': ticker, 'token': self.finnhub_key}
            
            response = self.session.get(url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if 'c' not in data or data['c'] == 0:
                return None
            
            current_price = data['c']
            prev_close = data['pc']
            change_pct = ((current_price - prev_close) / prev_close * 100) if prev_close != 0 else 0
            
            # Get company info
            company_url = f"https://finnhub.io/api/v1/stock/profile2"
            company_params = {'symbol': ticker, 'token': self.finnhub_key}
            company_response = self.session.get(company_url, params=company_params, timeout=10)
            company_data = company_response.json() if company_response.ok else {}
            
            return StockData(
                ticker=ticker,
                company_name=company_data.get('name', ticker),
                price=current_price,
                change_pct=change_pct,
                change_amount=current_price - prev_close,
                previous_close=prev_close,
                timestamp=datetime.now(timezone.utc),
                volume=0,  # Would need separate call
                market_cap=int(company_data.get('marketCapitalization', 0) * 1000000),
                currency=company_data.get('currency', 'USD'),
                source='finnhub'
            )
            
        except Exception as e:
            self.logger.error(f"Finnhub error for {ticker}: {e}")
            return None
    
    def generate_mock_data(self, ticker: str) -> StockData:
        """Generate mock data for testing when all sources fail"""
        import random
        
        base_price = random.uniform(50, 500)
        change_pct = random.uniform(-5, 5)
        prev_close = base_price / (1 + change_pct/100)
        
        return StockData(
            ticker=ticker,
            company_name=f"{ticker} Corporation",
            price=base_price,
            change_pct=change_pct,
            change_amount=base_price - prev_close,
            previous_close=prev_close,
            timestamp=datetime.now(timezone.utc),
            volume=random.randint(1000000, 50000000),
            market_cap=random.randint(1000000000, 100000000000),
            currency='USD',
            source='mock'
        )
    
    def fetch_stock_data(self, ticker: str, use_mock_fallback: bool = True) -> Optional[StockData]:
        """
        Fetch stock data with multiple source fallbacks
        
        Args:
            ticker: Stock symbol
            use_mock_fallback: Whether to use mock data if all sources fail
            
        Returns:
            StockData or None
        """
        # List of fetch methods to try in order
        fetch_methods = [
            self.fetch_from_yfinance,
            self.fetch_from_alpha_vantage,
            self.fetch_from_finnhub
        ]
        
        for method in fetch_methods:
            try:
                data = method(ticker)
                if data:
                    self.logger.info(f"Successfully fetched {ticker} from {data.source}")
                    return data
                else:
                    self.logger.warning(f"No data returned from {method.__name__}")
            except Exception as e:
                self.logger.error(f"Error in {method.__name__}: {e}")
            
            # Small delay between attempts
            time.sleep(1)
        
        # If all sources fail and mock is enabled
        if use_mock_fallback:
            self.logger.warning(f"All sources failed for {ticker}, using mock data")
            return self.generate_mock_data(ticker)
        
        return None
    
    def fetch_multiple_stocks(self, tickers: List[str]) -> Dict[str, StockData]:
        """Fetch data for multiple tickers"""
        results = {}
        for ticker in tickers:
            data = self.fetch_stock_data(ticker)
            if data:
                results[ticker] = data
            time.sleep(0.5)  # Rate limiting
        return results

class ContentGenerator:
    """Generates social media content for stock updates"""
    
    def __init__(self):
        self.logger = logging.getLogger(self.__class__.__name__)
    
    def generate_post_content(self, stock_data: StockData) -> str:
        """Generate social media post content"""
        # Determine trend emoji and sentiment
        if stock_data.change_pct > 2:
            trend_emoji = "🚀"
            sentiment = "soaring"
        elif stock_data.change_pct > 0:
            trend_emoji = "📈"
            sentiment = "rising"
        elif stock_data.change_pct < -2:
            trend_emoji = "📉"
            sentiment = "dropping"
        elif stock_data.change_pct < 0:
            trend_emoji = "⬇️"
            sentiment = "declining"
        else:
            trend_emoji = "➡️"
            sentiment = "stable"
        
        # Format price and change
        price_str = f"${stock_data.price:.2f}"
        change_str = f"{stock_data.change_pct:+.2f}%"
        
        # Create engaging content
        templates = [
            f"{trend_emoji} ${stock_data.ticker} is {sentiment} at {price_str} ({change_str})\n\n💰 Market Cap: ${stock_data.market_cap/1e9:.1f}B\n📊 Volume: {stock_data.volume:,}\n\n#Stocks #{stock_data.ticker} #Investing",
            
            f"📊 {stock_data.company_name} ({stock_data.ticker})\n{trend_emoji} {price_str} ({change_str})\n\n{sentiment.title()} today with volume of {stock_data.volume:,}\n\n#StockAlert #{stock_data.ticker}",
            
            f"{trend_emoji} Stock Update: ${stock_data.ticker}\n\nCurrent Price: {price_str}\nDaily Change: {change_str}\nVolume: {stock_data.volume:,}\n\n#MarketUpdate #Stocks #{stock_data.ticker}"
        ]
        
        # Select template based on change magnitude
        template_idx = min(abs(int(stock_data.change_pct)), len(templates) - 1)
        return templates[template_idx]

class StockBot:
    """Main bot orchestrator"""
    
    def __init__(self, tickers: List[str], post_threshold: float = 0.5):
        self.tickers = tickers
        self.post_threshold = post_threshold
        self.logger = logging.getLogger(self.__class__.__name__)
        
        # Initialize components
        self.fetcher = MultiSourceStockFetcher()
        self.content_generator = ContentGenerator()
        self.db = DatabaseManager()
    
    def should_post(self, stock_data: StockData, ticker: str) -> bool:
        """Determine if we should post based on change threshold and timing"""
        # Check if change is significant enough
        if abs(stock_data.change_pct) < self.post_threshold:
            self.logger.info(f"{ticker}: Change {stock_data.change_pct:.2f}% below threshold {self.post_threshold}%")
            return False
        
        # Check if we posted recently (within 6 hours)
        last_post_time = self.db.get_last_post_time(ticker)
        if last_post_time:
            hours_since_post = (datetime.now(timezone.utc) - last_post_time).total_seconds() / 3600
            if hours_since_post < 6:
                self.logger.info(f"{ticker}: Posted {hours_since_post:.1f} hours ago, waiting...")
                return False
        
        return True
    
    def run_single_cycle(self) -> Dict[str, Dict]:
        """Run a single bot cycle for all tickers"""
        results = {}
        
        for ticker in self.tickers:
            try:
                self.logger.info(f"Processing {ticker}...")
                
                # Fetch stock data
                stock_data = self.fetcher.fetch_stock_data(ticker)
                if not stock_data:
                    results[ticker] = {"success": False, "error": "Failed to fetch data"}
                    continue
                
                # Save to database
                self.db.save_stock_data(stock_data)
                
                # Check if we should post
                if not self.should_post(stock_data, ticker):
                    results[ticker] = {"success": True, "action": "skipped", "reason": "threshold/timing"}
                    continue
                
                # Generate content
                content = self.content_generator.generate_post_content(stock_data)
                
                # For now, just log the content (you'll add actual posting later)
                self.logger.info(f"Generated content for {ticker}:\n{content}")
                
                # Record successful "post"
                self.db.save_post_record(ticker, content, "", "console", True)
                
                results[ticker] = {
                    "success": True,
                    "action": "posted",
                    "data": stock_data.to_dict(),
                    "content": content
                }
                
            except Exception as e:
                self.logger.error(f"Error processing {ticker}: {e}")
                results[ticker] = {"success": False, "error": str(e)}
        
        return results

def main():
    """Main function to run the stock bot"""
    # Configuration
    TICKERS = ["AAPL", "GOOGL", "MSFT", "TSLA", "AMZN"]
    POST_THRESHOLD = 1.0  # Only post if change is >= 1%
    
    # Create bot instance
    bot = StockBot(tickers=TICKERS, post_threshold=POST_THRESHOLD)
    
    # Run single cycle
    logger.info("Starting stock bot cycle...")
    results = bot.run_single_cycle()
    
    # Print summary
    print("\n" + "="*50)
    print("STOCK BOT CYCLE SUMMARY")
    print("="*50)
    
    for ticker, result in results.items():
        print(f"\n{ticker}:")
        if result["success"]:
            if result["action"] == "posted":
                print(f"  ✅ POSTED - Change: {result['data']['change_pct']:+.2f}%")
                print(f"     Price: ${result['data']['price']:.2f}")
            else:
                print(f"  ⏸️  SKIPPED - {result['reason']}")
        else:
            print(f"  ❌ ERROR - {result['error']}")
    
    print("\n" + "="*50)

if __name__ == "__main__":
    main()

INFO:__main__:Starting stock bot cycle...
INFO:StockBot:Processing AAPL...
INFO:MultiSourceStockFetcher:Fetching AAPL from yfinance...
ERROR:yfinance:Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:AAPL: No price data found, symbol may be delisted (period=5d)
ERROR:yfinance:Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:AAPL: No price data found, symbol may be delisted (period=2d)
ERROR:yfinance:Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:AAPL: No price data found, symbol may be delisted (period=1d)
INFO:StockBot:Generated content for AAPL:
📊 AAPL Corporation (AAPL)
📈 $242.00 (+1.09%)

Rising today with volume of 41,212,793

#StockAlert #AAPL
INFO:StockBot:Processing GOOGL...
INFO:MultiSourceStockFetcher:Fetching GOOGL from yfinance...
ERROR:yfinance:Failed to get ticker 'GOOGL' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:GOOGL


STOCK BOT CYCLE SUMMARY

AAPL:
  ✅ POSTED - Change: +1.09%
     Price: $242.00

GOOGL:
  ✅ POSTED - Change: -1.02%
     Price: $71.45

MSFT:
  ✅ POSTED - Change: -4.00%
     Price: $361.20

TSLA:
  ✅ POSTED - Change: +4.88%
     Price: $451.82

AMZN:
  ✅ POSTED - Change: -2.13%
     Price: $122.48

